# Getting started with wfdb-x

**wfdb-x** is a faithful Python implementation of the
[WFDB Software Package](https://physionet.org/content/wfdb/) (version 10.7.0),
the library and tools used to read and write the physiologic signal,
annotation and header files of [PhysioNet](https://physionet.org). It reads
and writes exactly the same files as the original C library.

This notebook shows how to read the header and the signals of a record.

**Installation**

```
pip install wfdb-x                # the package (NumPy is its only dependency)
pip install "wfdb-x[examples]"    # also matplotlib and Jupyter, for these notebooks
wfdbx --examples my-examples      # copy these notebooks to ./my-examples
```

In [ ]:
import numpy as np
import wfdbx

# Read records from the sample data shipped with wfdb-x (the record "100s"):
# every relative record name is read under the input directory.
wfdbx.wfdbx_set_input_dir(wfdbx.sample_data_dir())

print("wfdb-x", wfdbx.__version__, "- WFDB", wfdbx.wfdbversion())
print("input directory:", wfdbx.wfdbx_get_input_dir())

## Where records are found

A WFDB record is a set of files with a common name: a header (`100s.hea`),
one or more signal files (`100s.dat`) and annotation files (`100s.atr`).
Every relative name is read under the **input directory** (set above with
`wfdbx.wfdbx_set_input_dir`, or with the `WFDBX_INPUT_DIR` environment
variable; by default, the current directory) and written under the output
directory (`wfdbx.wfdbx_set_output_dir`, `WFDBX_OUTPUT_DIR`). A name given as
a full path (`/Users/...`), a `~` path or a URL is used as given; nothing else
is searched. The signal files named in a header are in the directory of that
header. The high-level functions also accept `input_dir=` for a single call,
e.g. `wfdbx.read_record("100s", input_dir=wfdbx.sample_data_dir())`. The sample record
`100s`, shipped with wfdb-x, is the first minute of record 100 of the
[MIT-BIH Arrhythmia Database](https://physionet.org/content/mitdb/): two ECG
leads sampled at 360 Hz.

## The header

`read_header` returns a `Header`: the sampling frequency, the number of
samples and of signals, the start time and date, the comments, and one
`Siginfo` object per signal (`siginfo`).

In [ ]:
hdr = wfdbx.read_header("100s")
print(hdr)
for s in hdr.siginfo:
    print(f"{s.desc:5s} format {s.fmt}, gain {s.gain:g} adu per unit, "
          f"{s.adcres}-bit ADC, baseline {s.baseline}, units {s.units!r}")

The header of this record does not state the physical units of its
signals (`units` is `None`). wfdb-x never assumes values that a record does
not give: it reports them as unspecified (here, PhysioNet documents the
units of this database as millivolts). See *No built-in signal parameters*
in the README.

## The signals

`read_record` reads a record, or part of it, into a NumPy array with one
column per signal. With `physical=True`, the samples are converted to
physical units using the gains and baselines of the header.

In [ ]:
rec = wfdbx.read_record("100s", physical=True)
print(rec)
print("sampling frequency:", rec.fs, "Hz")
print("signals:", rec.names, "units:", rec.units, "gains:", rec.gains)
print("duration:", len(rec.signals) / rec.fs, "s")

Times can be given as sample numbers or as WFDB time strings (`"0:10"` is 10
seconds, `"s3600"` is sample 3600). Signals can be selected by number or by
name.

In [ ]:
part = wfdbx.read_record("100s", channels=["MLII", "V5"],
                         sampfrom="0:10", sampto="0:13", physical=True)
t = (part.sampfrom + np.arange(len(part.signals))) / part.fs
print(part.signals.shape, "samples from", t[0], "s to", t[-1], "s")

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:  # the plots are optional
    plt = None
    print("matplotlib is not installed: pip install matplotlib "
          "(or pip install \"wfdb-x[examples]\")")

if plt:
    fig, axes = plt.subplots(len(part.names), 1, sharex=True, figsize=(9, 4))
    for k, ax in enumerate(axes):
        ax.plot(t, part.signals[:, k], linewidth=0.8)
        ax.set_ylabel(f"{part.names[k]}\n({part.units[k] or 'units not given'})")
    axes[-1].set_xlabel("time (s)")
    plt.tight_layout()
    plt.show()

## ADC units and physical units

Without `physical=True`, the samples are the integers stored in the signal
file (ADC units). The conversion to physical units is
`(sample - baseline) / gain`:

In [ ]:
raw = wfdbx.read_record("100s", sampto=5)
print(raw.signals)
print((raw.signals - raw.baselines) / raw.gains)